# 20 — Figure Selection (label-driven pre-pre-processing)

Thin notebook: it only **imports**, **calls** `src/figure_selection.py`, and **displays**.
It decides **which figures** enter the embedding pipeline using only the human labels. It does not open any image.

**Input:** Stage 04's join in `paths.labelled_root` (`joined/master_labels.xlsx`, `joined/master_figures.xlsx`).
**Rules:** the `selection:` block of `config.yaml`. The reason for each rule is in `eVTOL-Visual-Evaluation/docs/embedding_evaluation/SELECTION_DECISIONS.md`.
**Output:** `<paths.pipeline_root>/selection/`: `candidates.csv`, `funnel.csv`, `aircraft.parquet`, `coverage.csv`, `sets/<name>.csv`, `sets_union.csv` and `selection_summary.json`.

The steps:
1. Build the candidate table: every figure on file, with its own labels and its aircraft's labels.
2. Apply the fixed gates, the same for every set, which gives the funnel.
3. Build one figure set per strategy.
4. Check coverage, and split the aircraft into a select half and a report half.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config_loader import load_config
from src import figure_selection as fs

cfg = load_config()
master, figures = fs.load_master(cfg)
print(len(master), 'aircraft rows |', len(figures), 'figure rows')

## 1. Candidate table

In [ ]:
cand = fs.build_candidates(master, figures)
cand.head()

## 2. Fixed gates → funnel
Each figure keeps the **first** gate that removed it (`excluded_by`).

In [ ]:
cand, funnel = fs.apply_gates(cand, cfg)
funnel

In [ ]:
# the figures left after the gates, by their labels
elig = cand[cand['eligible']]
for col in ['acState', 'per', 'acSty', 'qualityFlag', 'is_main']:
    display(elig[col].value_counts(dropna=False).rename(col).to_frame().T)

## 3. Figure sets (one per strategy)
Sets hold at most one figure per aircraft, except `all`. If an aircraft has several candidate figures, the notebook picks in this order: the main image, then `tie_break_perspective`, then quality `clean`, then figure order.

In [ ]:
sets = fs.build_sets(cand, cfg)
pd.DataFrame({k: {'figures': len(v), 'aircraft': v['aircraft_uid'].nunique()} for k, v in sets.items()}).T

## 4. Coverage and split
Aircraft covered by each set, broken down by topType. Strategy comparisons should use the aircraft shared by the compared sets. The `split` column divides the aircraft so the best strategy is **chosen** on `select` and **reported** on `report`.

In [ ]:
aircraft = fs.aircraft_table(master, cand, cfg)
cov = fs.coverage(aircraft, sets)
display(fs.coverage_by_type(cov, list(sets)))
common = (cov[list(sets)] > 0).all(axis=1)
print('eligible aircraft:', len(aircraft), '| in every set:', int(common.sum()))
print(aircraft['split'].value_counts().to_dict())

## Save

In [ ]:
fs.save(cand, funnel, aircraft, sets, cov, cfg)